# Phase 5 — Diagnostic de la divergence KL

Isole la cause de la KL ≈110 du PPO (asymétrie policy/ref vs bruit de quantification).

## ⚠️ Dataset à attacher
- `adl-dpo-adapter` (depuis le notebook 01)

**Setup** : GPU T4 x1, Internet On. **Durée** : ~10 min (charge 2 modèles à la fois).

> Prérequis : avoir poussé `eval/kl_diagnostics.py` sur le repo GitHub.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U bitsandbytes transformers==4.46.3 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 datasets==3.2.0 sentence-transformers faiss-cpu \
    anthropic openai pyarrow==17.0.0 tqdm

In [ ]:
!rm -rf /kaggle/working/adl
!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl
%cd /kaggle/working/adl

In [ ]:
# Vérifie que les scripts corrigés sont bien dans le repo cloné.
# Si ça échoue : commit + push des 4 fichiers sur ton repo GitHub d'abord.
import os
for p in ["eval/kl_diagnostics.py","training/train_ppo_fixed.py",
          "eval/evaluate_ethics_fixed.py","eval/stats_analysis.py"]:
    assert os.path.isfile(p), f"MANQUANT: {p} — pousse les scripts corrigés sur GitHub."
print("Scripts corrigés présents.")

In [ ]:
# Décompresse l'adapter DPO (warm-start de la policy ET de la ref)
DPO_ZIP = "/kaggle/input/adl-dpo-adapter/dpo_model.zip"
import os, zipfile
os.makedirs("results/dpo_model", exist_ok=True)
with zipfile.ZipFile(DPO_ZIP) as z: z.extractall("results/dpo_model")
print("DPO:", os.listdir("results/dpo_model"))

In [ ]:
# Quelques prompts suffisent pour le diagnostic d'initialisation
!python data/prepare_preferences.py --n_pku 2000 --n_ultra 1000 \
    --out_path data/preferences.jsonl

In [ ]:
!python eval/kl_diagnostics.py \
    --dpo_adapter results/dpo_model \
    --data_path data/preferences.jsonl \
    --n_prompts 16 \
    --out results/kl_diagnostics.json

In [ ]:
import json
print(json.dumps(json.load(open("results/kl_diagnostics.json")), indent=2))

## Lecture

- **A_buggy** = ton setup actuel (policy avec `prepare_model_for_kbit_training` + dropout, ref sans).
- **B_fixed_symmetric** = traitement symétrique, dropout off.
- **C_bf16_ref** = référence en bf16.

`kl_self` (KL de la policy avec elle-même) **doit** valoir ~0 : sinon le calcul de log-probs est buggé.

Si **A ≫ B ≈ 0** → la KL ≈110 vient de l'asymétrie de traitement, **pas** du bruit de quantification. Reporte ces 3 valeurs dans la sous-section 5.3 du rapport.